# f002の256検査診断・提出不可

T4 GPU・Internet OFF・新しいprivate Notebook。既存Competition/encoder-code/headに、新しいdecoder Input一つを追加。前処理・encoder・fold0 epoch11の重みを保持。

学習は行わず、goldを含まない固定weakリストから256検査を診断します。`submission.csv`は作らず、Competitionへ提出しません。

In [ ]:
from pathlib import Path
import json, traceback, os
SUMMARY_PATH = Path('/kaggle/working/F002_STREAMING_SUMMARY.json')
startup = {'status': 'started', 'phase': 'load_code', 'public_lb': None, 'training_performed': False}
def save_startup():
    SUMMARY_PATH.write_text(json.dumps(startup, ensure_ascii=False, indent=2), encoding='utf-8')
save_startup()
try:
    from pathlib import Path
    import json, sys, zipfile
    INPUT = Path('/kaggle/input/datasets/rsraki/rsna-frozen-feature-inputs-v1')
    if not INPUT.is_dir():
        INPUT = Path('/kaggle/input/rsna-frozen-feature-inputs-v1')
    CODE_INPUT = INPUT / 'code'  # Edit code Input ONLY; never scan the Competition tree.
    if not CODE_INPUT.is_dir():
        raise RuntimeError('Set CODE_INPUT to the manually attached code Input directory')
    bundles = list(CODE_INPUT.rglob('rsna-frozen-features-code.zip'))
    if len(bundles) > 1:
        raise RuntimeError('Multiple code ZIPs found; select one immutable code Input')
    if len(bundles) == 1:
        CODE = Path('/kaggle/working/frozen-code')
        if CODE.exists():
            raise RuntimeError('Fresh kernel/output directory required')
        CODE.mkdir()
        with zipfile.ZipFile(bundles[0]) as archive:
            for name in archive.namelist():
                dest = (CODE / name).resolve()
                if not dest.is_relative_to(CODE.resolve()):
                    raise ValueError('Unsafe archive member')
            archive.extractall(CODE)
    else:
        roots = [p.parents[2] for p in CODE_INPUT.rglob('feature_contracts.py') if p.parent.name == 'rsna_knee' and p.parent.parent.name == 'src']
        if len(roots) != 1:
            raise RuntimeError('Attach exactly one feature code bundle, ZIP or extracted src layout')
        CODE = roots[0]
    sys.path.insert(0, str(CODE / 'src'))
    import torch
    assert torch.cuda.is_available(), 'CUDA GPU required'
    print(torch.__version__, torch.cuda.get_device_name(0))
    
    # Edit these attached Input paths. Internet OFF. No file is downloaded by this notebook.
    ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
    if not ROOT.exists():
        ROOT = Path('/kaggle/input/rsna-knee-abnormality-detection')
    ENCODER_REPO = INPUT / 'encoder/dinov2-source'
    ENCODER_WEIGHTS = INPUT / 'encoder/dinov2_vits14_pretrain.pth'
    PROVENANCE = INPUT / 'encoder/encoder-provenance.json'
    provenance = json.loads(PROVENANCE.read_text(encoding='utf-8-sig'))
    startup.update(phase='code_ready', gpu=torch.cuda.get_device_name(0))
    save_startup()
except Exception as error:
    startup.update(status='failed', error=str(error), traceback=traceback.format_exc())
    save_startup()
    raise


In [ ]:
"""Same frozen preprocessing/head, one study at a time, with local failure diagnostics."""

from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import shutil
import subprocess
import sys
import time
import traceback
from collections import Counter, defaultdict
from pathlib import Path

REQUIRED_SYNTAXES = (
    "1.2.840.10008.1.2.4.57",
    "1.2.840.10008.1.2.4.70",
    "1.2.840.10008.1.2.4.90",
    "1.2.840.10008.1.2.4.91",
)


def decoder_inventory():
    from pydicom.pixels import get_decoder

    result = {}
    for syntax in REQUIRED_SYNTAXES:
        decoder = get_decoder(syntax)
        result[syntax] = {
            "available": decoder.is_available,
            "plugins": list(decoder.available_plugins),
            "missing_dependencies": decoder.missing_dependencies,
        }
    return result


def install_offline_decoders(wheels, manifest_sha256):
    """Install pinned Linux wheels only, without network or changing NumPy/torch."""
    directory = Path(wheels)
    manifest_path = directory.parent / "decoder-manifest.json"

    def digest(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()

    if digest(manifest_path) != manifest_sha256:
        raise ValueError("Decoder manifest SHA256 mismatch")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if sys.platform != "linux" or sys.version_info[:2] != (3, 13):
        raise RuntimeError("Decoder wheels target Kaggle Linux Python3.13; rebuild for another runtime")
    for wheel in manifest["wheels"]:
        path = directory / wheel["filename"]
        if digest(path) != wheel["sha256"]:
            raise ValueError("Decoder wheel SHA256 mismatch")
    before = {name: importlib.metadata.version(name) for name in ("numpy", "torch", "pydicom")}
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(directory),
            *[f"{item['name']}=={item['version']}" for item in manifest["wheels"]],
        ],
        check=True,
    )
    after = {name: importlib.metadata.version(name) for name in before}
    if after != before:
        raise ValueError("Decoder install changed the model/preprocessing environment")
    return {"manifest_sha256": manifest_sha256, "packages": manifest["wheels"], "unchanged": before}


def predict_arrays(model, arrays, device):
    import torch

    with torch.inference_mode():
        logits = model(**{name: torch.from_numpy(value).unsqueeze(0).to(device) for name, value in arrays.items()})
        if logits.shape != (1, 12) or not torch.isfinite(logits).all():
            raise ValueError("Nonfinite or malformed streaming head output")
        return logits.float().cpu().sigmoid()[0].tolist()


def verify_compressed_phantoms(directory, manifest_sha256):
    """Exercise installed decoders on our synthetic pixels, never on patient data."""
    import numpy as np
    import pydicom

    directory = Path(directory)
    manifest_path = directory / "phantom-manifest.json"
    if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != manifest_sha256:
        raise ValueError("Synthetic decoder phantom manifest SHA256 mismatch")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_path = directory / manifest["expected_filename"]
    if hashlib.sha256(expected_path.read_bytes()).hexdigest() != manifest["expected_sha256"]:
        raise ValueError("Synthetic expected pixels SHA256 mismatch")
    expected = np.load(expected_path, allow_pickle=False)
    results = []
    for item in manifest["files"]:
        path = directory / item["filename"]
        if hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Synthetic compressed fixture SHA256 mismatch")
        dataset = pydicom.dcmread(path)
        plugin_pixels = pydicom.pixels.pixel_array(dataset, decoding_plugin="pylibjpeg")
        default_pixels = dataset.pixel_array
        if not np.array_equal(plugin_pixels, expected) or not np.array_equal(default_pixels, expected):
            raise ValueError("Compressed decoder changed synthetic pixel values")
        results.append({"transfer_syntax": item["syntax"], "default_and_pylibjpeg_exact_pixels": True})
    return results


def resource_snapshot(device):
    import torch

    result = {"free_disk_bytes": shutil.disk_usage(Path.cwd()).free}
    try:
        import resource

        # Kaggle Linux reports KiB; this process high-water mark includes host arrays.
        result["cpu_peak_rss_bytes"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    except ImportError:
        result["cpu_peak_rss_bytes"] = None
    if torch.device(device).type == "cuda" and torch.cuda.is_available():
        result.update(
            gpu_allocated_bytes=torch.cuda.memory_allocated(device),
            gpu_reserved_bytes=torch.cuda.memory_reserved(device),
            gpu_peak_reserved_bytes=torch.cuda.max_memory_reserved(device),
        )
    return result


def run_streaming(
    root,
    checkpoint_path,
    provenance,
    encoder_repo,
    encoder_weights,
    output,
    *,
    mode="test",
    weak_manifest=None,
    profile_limit=256,
    selection_seed=20261007,
    expected_checkpoint_sha256,
    expected_sources,
    expected_weak_sha256=None,
    require_decoders=True,
    device="cuda",
):
    import numpy as np
    import torch

    from rsna_knee import feature_imaging
    from rsna_knee.contracts import (
        ID,
        SERIES_COLUMNS,
        SERIES_ID,
        SUBMISSION_COLUMNS,
        TARGETS,
        dump_json,
        index_studies,
        read_csv,
        sha256,
        source_hashes,
        uid,
        validate_submission,
        write_csv,
    )
    from rsna_knee.feature_contracts import feature_contract
    from rsna_knee.feature_model import FrozenFeatureHead
    from rsna_knee.feature_runtime import load_head_checkpoint

    root, output = Path(root), Path(output)
    if output.exists():
        raise ValueError("Use a fresh output directory/kernel; preserve the previous diagnostics")
    output.mkdir(parents=True)
    summary_path = output / "F002_STREAMING_SUMMARY.json"
    summary = {
        "status": "started",
        "phase": "preflight",
        "mode": mode,
        "public_lb": None,
        "training_performed": False,
        "processed": 0,
        "selection_seed": selection_seed,
        "fallback_predictions": 0,
        "skipped_series": 0,
        "runtime": {name: importlib.metadata.version(name) for name in ("torch", "numpy", "pydicom", "Pillow")},
        "note": "Saved-run diagnostics only; private scoring logs are not exposed by Kaggle",
    }
    began = time.perf_counter()
    journal = output / "events.jsonl"
    original_read = feature_imaging.read_feature_windows
    original_encode = feature_imaging.encode_windows
    current = {}
    syntax_counts = Counter()

    def event(phase, **details):
        summary.update(
            phase=phase,
            current_context={**current, **details},
            elapsed_seconds=time.perf_counter() - began,
            resources=resource_snapshot(device),
        )
        dump_json(summary_path, summary)
        with journal.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps({"phase": phase, **current, **details}, ensure_ascii=False) + "\n")

    def observed_read(series_dir, settings):
        import pydicom

        current["series_uid"] = Path(series_dir).name
        current.pop("header", None)
        event("read_series_header")
        paths = sorted(Path(series_dir).glob("*.dcm"))
        if paths:
            header = pydicom.dcmread(paths[0], stop_before_pixels=True)
            syntax = str(getattr(header.file_meta, "TransferSyntaxUID", ""))
            syntax_counts[syntax] += 1
            current["header"] = {
                "transfer_syntax": syntax,
                "slices": len(paths),
                "rows": int(getattr(header, "Rows", 0)),
                "columns": int(getattr(header, "Columns", 0)),
                "number_of_frames": int(getattr(header, "NumberOfFrames", 1)),
                "pixel_spacing": [float(v) for v in getattr(header, "PixelSpacing", [])],
                "orientation_present": hasattr(header, "ImageOrientationPatient"),
                "position_present": hasattr(header, "ImagePositionPatient"),
            }
        event("decode_and_preprocess_series")
        result = original_read(series_dir, settings)
        event("series_preprocessed", windows=len(result[0]))
        return result

    def observed_encode(model, windows, config, device):
        event("encode_series", input_shape=list(windows.shape))
        result = original_encode(model, windows, config, device)
        event("series_encoded", centers=len(result))
        return result

    try:
        event("preflight")
        if mode not in {"test", "profile_train"}:
            raise ValueError("Unknown streaming mode")
        if torch.device(device).type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU required")
        if source_hashes() != expected_sources or sha256(checkpoint_path) != expected_checkpoint_sha256:
            raise ValueError("Code/checkpoint differs from the audited f002 baseline")
        checkpoint = load_head_checkpoint(checkpoint_path)
        config = checkpoint["config"]
        if (checkpoint["fold"], checkpoint["epoch"]) != (0, 11):
            raise ValueError("Expected fixed f002 fold0 BCE-selected epoch11")
        if feature_contract(config, provenance) != checkpoint["feature_contract"]:
            raise ValueError("Encoder/preprocessing contract changed")
        summary.update(
            checkpoint_sha256=expected_checkpoint_sha256,
            feature_fingerprint=checkpoint["feature_fingerprint"],
            decoders=decoder_inventory(),
        )
        event("decoder_preflight")
        if require_decoders and any(not entry["available"] for entry in summary["decoders"].values()):
            raise RuntimeError("JPEG Lossless/JPEG2000 decoder unavailable; attach the pinned offline decoder Input")
        split = "test" if mode == "test" else "train"
        _, studies = read_csv(root / f"{split}.csv", (ID,))
        all_ids = index_studies(studies)
        if not all_ids:
            raise ValueError("No studies")
        if mode == "profile_train":
            if weak_manifest is None or type(profile_limit) is not int or profile_limit < 1:
                raise ValueError("Profile requires a fixed weak manifest and positive limit")
            if expected_weak_sha256 is None or sha256(weak_manifest) != expected_weak_sha256:
                raise ValueError("Profile weak manifest differs from the fixed gold-excluded training list")
            _, weak_rows = read_csv(weak_manifest, (ID,))
            eligible = index_studies(weak_rows)
            if not eligible or not eligible.keys() <= all_ids.keys():
                raise ValueError("Profile manifest does not match train.csv")
            keys = sorted(eligible, key=lambda key: hashlib.sha256(f"{selection_seed}:{key}".encode()).hexdigest())[
                :profile_limit
            ]
            studies = [{ID: key} for key in keys]
            truth_path = output / "profile-studies.csv"
            write_csv(truth_path, (ID,), studies)
        else:
            studies = [{ID: row[ID]} for row in studies]
            truth_path = root / "test.csv"
        _, series = read_csv(root / f"{split}_series.csv", SERIES_COLUMNS)
        by_study, pairs = defaultdict(list), set()
        for row in series:
            pair = (uid(row[ID]), uid(row[SERIES_ID]))
            if pair in pairs or pair[0] not in all_ids:
                raise ValueError("Orphan/duplicate series")
            pairs.add(pair)
            by_study[pair[0]].append(row)
        summary.update(
            studies=len(studies),
            study_csv_sha256=sha256(truth_path),
            series_csv_sha256=sha256(root / f"{split}_series.csv"),
            report_required=False,
            intermediate_feature_files=0,
            gpu=torch.cuda.get_device_name(device) if torch.device(device).type == "cuda" else "synthetic CPU test",
        )
        event("load_encoder")
        encoder = feature_imaging.load_frozen_encoder(encoder_repo, encoder_weights, provenance, device)
        head = FrozenFeatureHead(config).to(device)
        head.load_state_dict(checkpoint["model"], strict=True)
        head.eval()
        feature_imaging.read_feature_windows = observed_read
        feature_imaging.encode_windows = observed_encode
        predictions = []
        for number, row in enumerate(studies, 1):
            current.clear()
            current["study_uid"] = row[ID]
            event("extract_study", study_number=number)
            arrays, _ = feature_imaging.extract_study(root, split, row[ID], by_study[row[ID]], config, encoder, device)
            for value in arrays.values():
                if not np.isfinite(value).all():
                    raise ValueError("Nonfinite extracted feature arrays")
            if not arrays["mask"].any() or (arrays["positions"] < 0).any() or (arrays["positions"] > 1).any():
                raise ValueError("Empty mask or invalid normalized physical position")
            event("predict_head")
            scores = predict_arrays(head, arrays, device)
            predictions.append(
                {ID: row[ID], **{target: str(value) for target, value in zip(TARGETS, scores, strict=True)}}
            )
            del arrays
            if torch.device(device).type == "cuda":
                torch.cuda.synchronize(device)
            summary.update(processed=number, transfer_syntax_counts=dict(syntax_counts))
            event("study_complete")
            if number == 1 or number % 16 == 0 or number == len(studies):
                print(f"Streaming {number}/{len(studies)}; elapsed={time.perf_counter() - began:.1f}s", flush=True)
        event("validate_csv")
        # Publish final submission only after complete coverage and finite probabilities.
        temporary = output / "predictions.partial.csv"
        write_csv(temporary, SUBMISSION_COLUMNS, predictions)
        sample_path = root / "sample_submission.csv"
        receipt = validate_submission(
            temporary, truth_path, sample_path if mode == "test" and sample_path.is_file() else None
        )
        destination = output.parent / "submission.csv" if mode == "test" else output / "profile_predictions.csv"
        if destination.exists():
            raise ValueError("Refuse to replace a previous submission CSV")
        temporary.replace(destination)
        summary.update(
            status="passed" if mode == "test" else "profile_complete_not_for_submission",
            phase="complete",
            receipt=receipt,
            submission_sha256=sha256(destination),
            csv_name=destination.name,
            elapsed_seconds=time.perf_counter() - began,
            resources=resource_snapshot(device),
            per_study_seconds=(time.perf_counter() - began) / len(studies),
        )
        dump_json(summary_path, summary)
        return summary
    except Exception as error:
        summary.update(
            status="failed",
            error_type=type(error).__name__,
            error=str(error),
            traceback=traceback.format_exc(),
            elapsed_seconds=time.perf_counter() - began,
        )
        dump_json(summary_path, summary)
        print(summary["traceback"], flush=True)
        raise
    finally:
        feature_imaging.read_feature_windows = original_read
        feature_imaging.encode_windows = original_encode
        gc.collect()


In [ ]:
DECODER_INPUT = Path('/kaggle/input/datasets/rsraki/rsna-dicom-decoders-py313-v1')
if not DECODER_INPUT.is_dir():
    DECODER_INPUT = Path('/kaggle/input/rsna-dicom-decoders-py313-v1')
try:
    startup['phase'] = 'offline_decoder_install'
    save_startup()
    if not DECODER_INPUT.is_dir():
        raise RuntimeError('Attach the new private rsna-dicom-decoders-py313-v1 Dataset; do not upload model weights again')
    decoder_install = install_offline_decoders(DECODER_INPUT / 'wheels', '0db631c509b9878b050d05a328d7479a0671f46db51959ec10b0b2fd73f2f1a5')
    startup['phase'] = 'synthetic_compressed_decoder_check'
    save_startup()
    decoder_install['synthetic_pixel_checks'] = verify_compressed_phantoms(DECODER_INPUT / 'phantoms', '882a942b99d5fb9a55e7521bcc24ec22edc76fba12f094cffb8afe3100a11dba')
    startup.update(decoder_install=decoder_install, decoders=decoder_inventory(), phase='decoders_ready')
    save_startup()
except Exception as error:
    startup.update(status='failed', error=str(error), traceback=traceback.format_exc())
    save_startup()
    raise


In [ ]:
HEAD_INPUT = Path('/kaggle/input/datasets/rsraki/rsna-f002-head-fold0-v1')
if not HEAD_INPUT.is_dir():
    HEAD_INPUT = Path('/kaggle/input/rsna-f002-head-fold0-v1')
MODE = 'profile_train'
PROFILE_LIMIT = 256
PROFILE_SEED = 20261007
DETAILS = Path('/kaggle/working/f002-streaming-diagnostics')
try:
    if MODE == 'profile_train' and os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        raise RuntimeError('Profile notebook is not for competition submission')
    result = run_streaming(
        ROOT, HEAD_INPUT / 'best.pt', provenance, ENCODER_REPO, ENCODER_WEIGHTS, DETAILS,
        mode=MODE, weak_manifest=INPUT / 'manifest/weak.csv' if MODE == 'profile_train' else None,
        profile_limit=PROFILE_LIMIT, selection_seed=PROFILE_SEED,
        expected_checkpoint_sha256='189ecab25e0d1c029e974ee23f3c46239e5eceef290b5dca33f6831dcb8e1b03', expected_sources={'__init__.py': 'a6f06061df2e890790f44c5ba4af472dd0abc6f465ad48044e6cee53ae0f7751', '__main__.py': '6d8b7d7846a845059d7a3107143f11131f63c5511d669b44085b15ec5e3d2279', 'checkpoints.py': '4ffc94da42c34798c263360ecdb4af7340f417d653cb7e37ad1b6169937be541', 'cli.py': 'e8d1a1267abee34101be88f744938aecb3e2f27fa4e15fd4747c3baca8d16cee', 'contracts.py': 'aa246fd1a49ecbe88f1795f49904ef9675f119c935e99b9ffd5eee8220cbc71b', 'diagnostics.py': 'aa89c9f29a8c641d005405b306f060a782c864d3e35a3b3371c7d3ffb4c1428a', 'feature_contracts.py': 'be76ecc28b14491f4d1a0a459794c7f94456e845ce0e46c46ced699bfaf54ed6', 'feature_imaging.py': 'de36a21929b482eee897b12e06bb9964cee59beab6a9bc2b2d44c15ace279546', 'feature_model.py': 'af85d8eaed6015ed2ddb7f0b2029eca446471876750ebde60bed403cf8ced0e1', 'feature_runtime.py': 'bfa9c7d075f342bd45fcdf2152e7560587972923a1e37e6d915f7e617399e01e', 'imaging.py': '07a114888cde19ae0e5799f2de48c6751ec9c4b5637ec1f4f4c220f2911db9ff', 'inputs.py': '2399ebaaf1f1c9fe3e3bcc302c90d2afa8f81c5352e16f5af6ae03efe93ff87f', 'metrics.py': 'c5f3944e55392ae8a98fd6ed79b140a95d7af58fc3d3537d41ceefcc4c3b4f1c', 'model.py': '86d21b96701c7ca99231fce3ddbd8f8d1c0967d81059b42787ef831cf203a96a', 'prepare.py': 'da481889af0ebe9528ff2f150e78d35c9123f2d50e475b11167b785fbd90920e', 'runtime.py': '09c7ee998e1c3be02f0ed3f00db6acc92351ea374fcf8a469c8d4fcc57233f05'},
        expected_weak_sha256='700e3da8016d80fd87bde2d1c189ef755ce5e5f89717ebac44d011c967a8371a', device='cuda')
    result['decoder_install'] = decoder_install
    SUMMARY_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(result, ensure_ascii=False, indent=2))
except Exception as error:
    saved = DETAILS / 'F002_STREAMING_SUMMARY.json'
    result = json.loads(saved.read_text(encoding='utf-8')) if saved.exists() else dict(startup)
    result.update(status='failed', error=str(error), traceback=traceback.format_exc(), decoder_install=decoder_install)
    SUMMARY_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    raise
